# Final Comparison

In [1]:
import utils
import json
import numpy as np
import os
from utils import postprocess_responses
import math    
from collections import defaultdict
from collections import Counter
from datasets import load_dataset, load_from_disk

np.random.seed(42)

c:\Users\nanfangwuyu\.conda\envs\RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [70]:
import importlib
importlib.reload(utils)

<module 'utils' from 'c:\\Users\\nanfangwuyu\\MyDrive\\WorkSpace\\CodeWorkSpace\\Master Thesis\\Master-Thesis-Beyond-Answer\\VLM CoT Task\\utils.py'>

In [2]:
# ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
ds = load_from_disk('datasets/ARC-Easy-processed2')
option_s = 'options' if 'options' in ds[0] else 'choices'

In [3]:
MODEL_LIST = [
    "qwen3-235b-a22b-thinking-2507",
    "qwen3-30b-a3b-thinking-2507",
    "qwen3-4b",
]
# each 2 models, each 3 models and all 4 models
MODEL_LIST_ALL = [(MODEL_LIST[i], MODEL_LIST[j]) for i in range(len(MODEL_LIST)) for j in range(i+1, len(MODEL_LIST))] + \
                 [(MODEL_LIST[i], MODEL_LIST[j], MODEL_LIST[k]) for i in range(len(MODEL_LIST)) for j in range(i+1, len(MODEL_LIST)) for k in range(j+1, len(MODEL_LIST))] 

settings = ['cot_trigger', 'cot_trigger', 'cot_trigger'] 
settings_all = [(settings[i], settings[j]) for i in range(len(settings)) for j in range(i+1, len(settings))] + \
                 [(settings[i], settings[j], settings[k]) for i in range(len(settings)) for j in range(i+1, len(settings)) for k in range(j+1, len(settings))] 
DATASET_NAME = 'ARC-Easy'
N = 16

In [4]:
force_reprocess = True

def run(MODEL_LIST=MODEL_LIST, settings=settings):
    np.random.seed(42)
    multi_cot_responses = []
    multi_file_names = []
    metas = []
    for MODEL_NAME, setting in zip(MODEL_LIST, settings):
        metas.append((MODEL_NAME, setting))
        cot_responses_list, file_names = utils.load_latest_response(f'data/n_vs_one/{DATASET_NAME}/{MODEL_NAME}/responses_{MODEL_NAME}_{DATASET_NAME}_{setting}', N, debug=False)
        cot_responses = [utils.clean_responses(item, mode=MODEL_NAME, setting=setting) for item in cot_responses_list]
        file_names = [file_name.split('_')[-1].split('.')[0] for file_name in file_names]

        multi_cot_responses.append(cot_responses)
        multi_file_names.append(file_names)
    print(N, min(len(x) for x in multi_file_names))
    assert N == min(len(x) for x in multi_file_names)
    assert len(ds) == len(multi_cot_responses[0][0])
    multi_lst_AE_all, multi_match_list_all = [], []
    for i, meta in enumerate(metas):
        MODEL_NAME, setting = meta
        print(MODEL_NAME, setting)
        lst_AE_all, match_list_all = postprocess_responses(ds, DATASET_NAME, MODEL_NAME, multi_cot_responses[i], multi_file_names[i], force_reprocess=force_reprocess)
        multi_lst_AE_all.append(lst_AE_all)
        multi_match_list_all.append(match_list_all)
    option_lengths = [len(ops) for ops in ds[option_s]]

    def calculate_entropy_distribution(ans_dist, n):

        def cal_entropy_scipy(answer_list, len_ops):
            from scipy.stats import entropy

            # count for i in range (0, len_ops), each appears how many times
            counts = {i: np.sum(answer_list == i) for i in range(len_ops)}
            pk = np.array(list(counts.values()), dtype=float)
            # pk = [9,1,0,0,0]
            # print(pk)
            n = len_ops  # number of discrete categories
            H = entropy(pk, base=2)           
            # print(H)   # entropy in bits
            H_norm = H / (np.log2(n))              # normalized to max entropy of log2(n)

            max_count = max(counts.values())
            max_keys = [k for k, v in counts.items() if v == max_count]
            best_choice = np.random.choice(max_keys)
            return H_norm, best_choice

        entropy_distribution = []
        choice_list = []
        for j in range(len(ans_dist)):
            ans_dist_j = ans_dist[j][:n]
            # len_ops = len(ds[option_s][j])
            # entropy, best_choice = cal_entropy(ans_dist_j)
            entropy, best_choice = cal_entropy_scipy(ans_dist_j, len_ops=option_lengths[j])
            entropy_distribution.append(entropy)
            choice_list.append(best_choice)
        return entropy_distribution, choice_list

    def calculate_majority_distribution(ans_dist, n):
        
        def cal_majority(answer_list):

            counts = defaultdict(int)
            for answer in answer_list:
                if 0 <= answer <= 9:
                    counts[answer] += 1
                else:
                    print(f"Error! Idx {answer}")

            # Find the majority choice (the most frequent answer)
            max_count = max(counts.values())
            max_keys = [k for k, v in counts.items() if v == max_count]
            majority_choice = np.random.choice(max_keys)
            return majority_choice

        choice_list = []
        for j in range(len(ans_dist)):
            ans_dist_j = ans_dist[j][:n]
            majority_choice = cal_majority(ans_dist_j)
            # If there is at least one correct answer
            choice_list.append(majority_choice)
        return choice_list

    def calculate_accuracy(choice_list, idx_ground_truth):
        correct_count = sum(1 for i, choice in enumerate(choice_list) if choice == idx_ground_truth[i])
        accuracy = correct_count / len(idx_ground_truth)
        return accuracy

    def calculate_match(choice_list, idx_ground_truth):
        match_list = [1 if choice == idx_ground_truth[i] else 0 for i, choice in enumerate(choice_list)]
        return match_list

    idx_ground_truth = []
    for i, ans in enumerate(ds['answer']):
        if DATASET_NAME in ["MathVista"]:
            idx_ground_truth.append(ds[option_s][i].index(ans))
        elif DATASET_NAME in ["ScienceQA", "TQA"]:
            idx_ground_truth.append(ans)
        else:
            chara_list = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
            idx_ground_truth.append(chara_list.index(ans))
            
    multi_lst_AE_all = np.array(multi_lst_AE_all)
    multi_ent_dist_all = []
    multi_ent_choice_all = []
    multi_ent_acc_all = []
    multi_ent_match_all = []
    for ans_dist in multi_lst_AE_all:
        ent_dist_all = []
        ent_choice_all = []
        ent_acc_all = []
        ent_match_all = []
        for n in range(1, N + 1):
            entropy_distribution, entropy_choices = calculate_entropy_distribution(ans_dist.T, n)
            ent_dist_all.append(entropy_distribution)
            ent_choice_all.append(entropy_choices)
            ent_acc_all.append(calculate_accuracy(entropy_choices, idx_ground_truth))
            ent_match_all.append(calculate_match(entropy_choices, idx_ground_truth))
        multi_ent_dist_all.append(ent_dist_all)
        multi_ent_choice_all.append(ent_choice_all)
        multi_ent_acc_all.append(ent_acc_all)
        multi_ent_match_all.append(ent_match_all)

    multi_major_choice_all = []
    multi_major_acc_all = []

    for i, ans_dist in enumerate(multi_lst_AE_all):
        major_choice_all = []
        major_acc_all = []
        for n in range(1, N + 1):
            major_choices = calculate_majority_distribution(ans_dist.T, n)
            major_choice_all.append(major_choices)
            major_acc_all.append(calculate_accuracy(major_choices, idx_ground_truth))
        multi_major_choice_all.append(major_choice_all)
        multi_major_acc_all.append(major_acc_all)
        
    multi_cot_responses = np.array(multi_cot_responses, dtype=object)  # shape: (3, 16, 540)
    multi_lst_AE_all = np.array(multi_lst_AE_all)  # shape: (3, 16, 540)
    multi_ent_dist_all = np.array(multi_ent_dist_all)  # shape: (3, 16, 540)
    multi_ent_choice_all = np.array(multi_ent_choice_all)  # shape: (3, 16, 540)
    multi_ent_acc_all = np.array(multi_ent_acc_all) # shape: (3, 16)
    multi_ent_match_all = np.array(multi_ent_match_all)  # shape: (3, 16, 540)
    multi_major_choice_all = np.array(multi_major_choice_all)  # shape: (3, 16, 540)
    multi_major_acc_all = np.array(multi_major_acc_all)  # shape: (3, 16)
    multi_match_list_all = np.array(multi_match_list_all)  # shape: (3, 16, 540)

    def cal_multi_round_avg_acc(n, debug=False):
        accs_list = []
        for i in range(len(metas)):
            accs = np.mean(multi_match_list_all[i][:n], axis=1)
            if debug:
                print(accs)
            accs_list.append(np.mean(accs))
        return accs_list

    def cal_multi_model_avg_acc(n):
        accs_all = []
        for i in range(len(metas)):
            accs = np.mean(multi_match_list_all[i][:n], axis=1)
            accs_all.append(np.mean(accs))
        return np.mean(accs_all)

    def cal_multi_model_major_acc(n):
        multi_major_choice_all_T = multi_major_choice_all.T  # shape: (540, 16, 3)
        final_choice_list = []
        for j in range(len(multi_major_choice_all_T)):
            # 当前题目在 n 轮之后，多模型给出的 choice 列表
            choices = multi_major_choice_all_T[j][n-1]
            choice_counter = Counter(choices)

            # 找到最大频数
            max_count = max(choice_counter.values())
            # 所有达到最大频数的候选
            max_keys = [k for k, v in choice_counter.items() if v == max_count]
            # 在这些并列最多的候选里随机选一个
            final_choice = np.random.choice(max_keys)

            final_choice_list.append(final_choice)

        final_acc = calculate_accuracy(final_choice_list, idx_ground_truth)
        print(f"Majority Voting Accuracy: {final_acc}")
        return final_acc

    def cal_multi_model_ent_acc_no_prior(n):
        multi_ent_choice_all_T = multi_ent_choice_all.T  # shape: (540, 16, 3)
        multi_ent_dist_all_T = multi_ent_dist_all.T      # shape: (540, 16, 3)
        final_choice_list = []
        for j in range(len(multi_ent_choice_all_T)):
            entropy_distribution = multi_ent_dist_all_T[j][n-1]      # 这一题、这一轮，各模型的熵
            choices_this_round = multi_ent_choice_all_T[j][n-1]      # 这一题、这一轮，各模型的选择

            # 找到最小熵
            min_entropy = np.min(entropy_distribution)
            # 所有达到最小熵的模型索引
            min_indices = np.where(entropy_distribution == min_entropy)[0]
            # 在这些最小熵的模型中随机选一个模型
            chosen_model_idx = np.random.choice(min_indices)

            # 取该模型给出的选项
            final_choice = choices_this_round[chosen_model_idx]
            final_choice_list.append(final_choice)

        final_acc = calculate_accuracy(final_choice_list, idx_ground_truth)
        print(f"Entropy Voting Accuracy: {final_acc}")
        return final_acc

    def pipe(n=N):
        results = {
            'multi_round': {},
            'multi_model': {}}

        # 多模型
        avg_accs = results['multi_round']['avg_acc'] = cal_multi_round_avg_acc(n)
        print(avg_accs)
        results['multi_model']['min_acc'] = min(avg_accs)
        results['multi_model']['max_acc'] = max(avg_accs)
        results['multi_model']['avg_acc'] = cal_multi_model_avg_acc(n)
        results['multi_model']['voting_acc'] = cal_multi_model_major_acc(n)
        results['multi_model']['ent_no_prior_acc'] = cal_multi_model_ent_acc_no_prior(n)
        return results

    # 运行管道
    results = pipe()
    print("\nFinal Results:")
    for key, value in results.items():
        for sub_key, sub_value in value.items():
            print(f"{key} - {sub_key}: {sub_value}")


    di = dict(zip(MODEL_LIST, results['multi_round']['avg_acc']))
    # add results['multi_model'] to di
    di.update(results['multi_model'])
    import pandas as pd
    display(pd.DataFrame(di, index=[DATASET_NAME]))


In [59]:
run(MODEL_LIST=MODEL_LIST[1:], settings=settings[1:])

16 16
qwen3-30b-a3b-thinking-2507 cot_trigger
qwen3-4b cot_trigger
[np.float64(0.5946443602693603), np.float64(0.9450494528619529)]
Majority Voting Accuracy: 0.7849326599326599
Entropy Voting Accuracy: 0.9478114478114478

Final Results:
multi_round - avg_acc: [np.float64(0.5946443602693603), np.float64(0.9450494528619529)]
multi_model - min_acc: 0.5946443602693603
multi_model - max_acc: 0.9450494528619529
multi_model - avg_acc: 0.7698469065656566
multi_model - voting_acc: 0.7849326599326599
multi_model - ent_no_prior_acc: 0.9478114478114478


,qwen3-30b-a3b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.594644,0.945049,0.594644,0.945049,0.769847,0.784933,0.947811


In [75]:
run(MODEL_LIST=MODEL_LIST[1:], settings=settings[1:])

16 16
qwen3-30b-a3b-thinking-2507 cot_trigger
qwen3-4b cot_trigger
[np.float64(0.9717487373737375), np.float64(0.9599905303030303)]
Majority Voting Accuracy: 0.9789562289562289
Entropy Voting Accuracy: 0.9869528619528619

Final Results:
multi_round - avg_acc: [np.float64(0.9717487373737375), np.float64(0.9599905303030303)]
multi_model - min_acc: 0.9599905303030303
multi_model - max_acc: 0.9717487373737375
multi_model - avg_acc: 0.9658696338383839
multi_model - voting_acc: 0.9789562289562289
multi_model - ent_no_prior_acc: 0.9869528619528619


,qwen3-30b-a3b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.971749,0.959991,0.959991,0.971749,0.96587,0.978956,0.986953


In [7]:
MODEL_LIST = ['qwen3-4b', 'qwen3-30b-a3b-thinking-2507']
settings = ['cot_trigger', 'cot_trigger']

In [8]:
np.random.seed(42)
force_reprocess = False
multi_cot_responses = []
multi_file_names = []
metas = []
for MODEL_NAME, setting in zip(MODEL_LIST, settings):
    metas.append((MODEL_NAME, setting))
    cot_responses_list, file_names = utils.load_latest_response(f'data/n_vs_one/{DATASET_NAME}/{MODEL_NAME}/responses_{MODEL_NAME}_{DATASET_NAME}_{setting}', N, debug=False)
    cot_responses = [utils.clean_responses(item, mode=MODEL_NAME, setting=setting) for item in cot_responses_list]
    file_names = [file_name.split('_')[-1].split('.')[0] for file_name in file_names]

    multi_cot_responses.append(cot_responses)
    multi_file_names.append(file_names)
print(N, min(len(x) for x in multi_file_names))
assert N == min(len(x) for x in multi_file_names)
assert len(ds) == len(multi_cot_responses[0][0])
multi_lst_AE_all, multi_match_list_all = [], []
for i, meta in enumerate(metas):
    MODEL_NAME, setting = meta
    print(MODEL_NAME, setting)
    lst_AE_all, match_list_all = postprocess_responses(ds, DATASET_NAME, MODEL_NAME, multi_cot_responses[i], multi_file_names[i], force_reprocess=force_reprocess)
    multi_lst_AE_all.append(lst_AE_all)
    multi_match_list_all.append(match_list_all)
option_lengths = [len(ops) for ops in ds[option_s]]

16 16
qwen3-4b cot_trigger
qwen3-30b-a3b-thinking-2507 cot_trigger


In [13]:
def calculate_entropy_distribution(ans_dist, n):

        def cal_entropy_scipy(answer_list, len_ops):
            from scipy.stats import entropy

            # count for i in range (0, len_ops), each appears how many times
            counts = {i: np.sum(answer_list == i) for i in range(len_ops)}
            pk = np.array(list(counts.values()), dtype=float)
            # pk = [9,1,0,0,0]
            # print(pk)
            n = len_ops  # number of discrete categories
            H = entropy(pk, base=2)           
            # print(H)   # entropy in bits
            H_norm = H / (np.log2(n))              # normalized to max entropy of log2(n)

            max_count = max(counts.values())
            max_keys = [k for k, v in counts.items() if v == max_count]
            best_choice = np.random.choice(max_keys)
            return H_norm, best_choice

        entropy_distribution = []
        choice_list = []
        for j in range(len(ans_dist)):
            ans_dist_j = ans_dist[j][:n]
            # len_ops = len(ds[option_s][j])
            # entropy, best_choice = cal_entropy(ans_dist_j)
            entropy, best_choice = cal_entropy_scipy(ans_dist_j, len_ops=option_lengths[j])
            entropy_distribution.append(entropy)
            choice_list.append(best_choice)
        return entropy_distribution, choice_list

def calculate_majority_distribution(ans_dist, n):
    
    def cal_majority(answer_list):

        counts = defaultdict(int)
        for answer in answer_list:
            if 0 <= answer <= 9:
                counts[answer] += 1
            else:
                print(f"Error! Idx {answer}")

        # Find the majority choice (the most frequent answer)
        max_count = max(counts.values())
        max_keys = [k for k, v in counts.items() if v == max_count]
        majority_choice = np.random.choice(max_keys)
        return majority_choice

    choice_list = []
    for j in range(len(ans_dist)):
        ans_dist_j = ans_dist[j][:n]
        majority_choice = cal_majority(ans_dist_j)
        # If there is at least one correct answer
        choice_list.append(majority_choice)
    return choice_list

def calculate_accuracy(choice_list, idx_ground_truth):
    correct_count = sum(1 for i, choice in enumerate(choice_list) if choice == idx_ground_truth[i])
    accuracy = correct_count / len(idx_ground_truth)
    return accuracy

def calculate_match(choice_list, idx_ground_truth):
    match_list = [1 if choice == idx_ground_truth[i] else 0 for i, choice in enumerate(choice_list)]
    return match_list

idx_ground_truth = []
for i, ans in enumerate(ds['answer']):
    if DATASET_NAME in ["MathVista"]:
        idx_ground_truth.append(ds[option_s][i].index(ans))
    elif DATASET_NAME in ["ScienceQA", "TQA"]:
        idx_ground_truth.append(ans)
    else:
        chara_list = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
        idx_ground_truth.append(chara_list.index(ans))
        
multi_lst_AE_all = np.array(multi_lst_AE_all)
multi_ent_dist_all = []
multi_ent_choice_all = []
multi_ent_acc_all = []
multi_ent_match_all = []
for ans_dist in multi_lst_AE_all:
    ent_dist_all = []
    ent_choice_all = []
    ent_acc_all = []
    ent_match_all = []
    for n in range(1, N + 1):
        entropy_distribution, entropy_choices = calculate_entropy_distribution(ans_dist.T, n)
        ent_dist_all.append(entropy_distribution)
        ent_choice_all.append(entropy_choices)
        ent_acc_all.append(calculate_accuracy(entropy_choices, idx_ground_truth))
        ent_match_all.append(calculate_match(entropy_choices, idx_ground_truth))
    multi_ent_dist_all.append(ent_dist_all)
    multi_ent_choice_all.append(ent_choice_all)
    multi_ent_acc_all.append(ent_acc_all)
    multi_ent_match_all.append(ent_match_all)

multi_major_choice_all = []
multi_major_acc_all = []

for i, ans_dist in enumerate(multi_lst_AE_all):
    major_choice_all = []
    major_acc_all = []
    for n in range(1, N + 1):
        major_choices = calculate_majority_distribution(ans_dist.T, n)
        major_choice_all.append(major_choices)
        major_acc_all.append(calculate_accuracy(major_choices, idx_ground_truth))
    multi_major_choice_all.append(major_choice_all)
    multi_major_acc_all.append(major_acc_all)
    
multi_cot_responses = np.array(multi_cot_responses, dtype=object)  # shape: (3, 16, 540)
multi_lst_AE_all = np.array(multi_lst_AE_all)  # shape: (3, 16, 540)
multi_ent_dist_all = np.array(multi_ent_dist_all)  # shape: (3, 16, 540)
multi_ent_choice_all = np.array(multi_ent_choice_all)  # shape: (3, 16, 540)
multi_ent_acc_all = np.array(multi_ent_acc_all) # shape: (3, 16)
multi_ent_match_all = np.array(multi_ent_match_all)  # shape: (3, 16, 540)
multi_major_choice_all = np.array(multi_major_choice_all)  # shape: (3, 16, 540)
multi_major_acc_all = np.array(multi_major_acc_all)  # shape: (3, 16)
multi_match_list_all = np.array(multi_match_list_all)  # shape: (3, 16, 540)

In [33]:
multi_match_list_all[0][0][:10], multi_match_list_all[1][0][:10]

(array([False,  True,  True,  True, False,  True, False,  True,  True,
         True]),
 array([ True, False,  True,  True, False, False, False, False,  True,
         True]))

In [35]:
print(idx_ground_truth)

[0, 1, 3, 3, 1, 2, 0, 2, 2, 0, 1, 1, 1, 1, 1, 1, 1, 3, 1, 3, 3, 3, 3, 2, 1, 2, 1, 1, 0, 1, 0, 2, 0, 0, 2, 0, 3, 2, 2, 1, 0, 1, 0, 2, 0, 3, 3, 2, 3, 3, 0, 1, 2, 0, 2, 1, 3, 2, 1, 0, 1, 3, 3, 1, 0, 0, 1, 1, 0, 1, 0, 3, 3, 1, 2, 3, 1, 1, 1, 0, 3, 0, 2, 0, 1, 0, 0, 0, 2, 3, 2, 2, 2, 1, 1, 0, 0, 2, 1, 2, 1, 3, 1, 2, 3, 2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 1, 1, 1, 0, 0, 0, 2, 1, 2, 2, 2, 0, 2, 2, 1, 2, 0, 1, 0, 1, 3, 0, 1, 3, 2, 1, 3, 1, 3, 0, 3, 0, 1, 1, 0, 0, 0, 3, 0, 3, 3, 3, 0, 3, 1, 3, 1, 2, 3, 2, 0, 1, 0, 0, 1, 2, 1, 3, 2, 2, 2, 0, 1, 2, 2, 1, 1, 0, 0, 2, 3, 1, 2, 1, 2, 0, 2, 0, 0, 1, 1, 2, 2, 3, 1, 2, 2, 2, 0, 1, 1, 2, 3, 2, 1, 1, 2, 2, 0, 0, 2, 0, 2, 1, 1, 3, 3, 2, 3, 3, 3, 0, 1, 3, 1, 0, 3, 1, 1, 1, 0, 3, 3, 0, 3, 1, 3, 3, 0, 2, 3, 1, 1, 0, 0, 2, 3, 1, 2, 3, 0, 3, 3, 1, 1, 0, 2, 1, 3, 2, 3, 0, 2, 2, 1, 2, 3, 0, 1, 0, 1, 1, 1, 2, 2, 0, 0, 2, 0, 0, 3, 3, 3, 0, 2, 1, 1, 2, 3, 0, 2, 0, 0, 1, 2, 2, 0, 1, 1, 2, 1, 2, 1, 2, 3, 1, 3, 0, 3, 1, 1, 3, 1, 1, 3, 0, 1, 0, 3, 2, 1, 2, 1, 0, 0, 3, 1, 0, 

In [56]:
multi_cot_responses[0][0][9][-100:], multi_cot_responses[1][0][9][-100:]

('nly correct statement is **[A]**.\n\n**Answer:** [A] Xylem carries water from the roots to the leaves.',
 'not phloem. Incorrect.\n\nThus, only option [A] accurately describes plant transport.\n\n**Answer: [A]**')

In [36]:
multi_lst_AE_all[0][0][:10], multi_lst_AE_all[1][0][:10]

(array([3, 1, 3, 3, 2, 2, 1, 2, 2, 0]), array([0, 0, 3, 3, 0, 3, 2, 0, 2, 0]))

array([ True, True,  True,  True, True, False, True, True, True, True]))
[0,1,3,3,1,0,2,2,0]

In [65]:
# ...existing code...
import re

def Answer_Extraction_Pipeline(resp, opts, debug=False, mode='sim', thre=0.8, using_pixtral=False):
    """
    resp:  模型完整输出（含 CoT）
    opts:  备选项列表（如 ['A. xxx', 'B. yyy', ...] 或 ['dog','cat',...]
    mode:  'sim' => 无阈值，直接返回最相似选项；否则如果置信度 < thre 返回 -1
    """
    # 允许的字母选项上限（可根据数据集加长）
    charas = [chr(ord('A') + i) for i in range(10)]  # A-J

    # ---------- Step 1: 提取 new_resp ----------
    new_resp = resp.strip().split("assistant")[-1]
    if using_pixtral:
        # 你之前对 pixtral 的特殊逻辑
        new_resp = resp.split('**Answer:**')[-1].strip()

    # ---------- Step 2: 提取 tail ----------
    def extract_tail(text: str):
        # 取最后 1~2 个句子作为 tail
        sentences = re.split(r'(?<=[.?!])\s+|\n', text.strip())
        if not sentences:
            return text.strip()
        tail_sentences = sentences[-2:] if len(sentences) >= 2 else sentences[-1:]
        return ' '.join(s.strip() for s in tail_sentences if s.strip())

    tail = extract_tail(new_resp)
    if debug:
        print(f"[Tail] {tail}")

    # ---------- Step 3: 从 tail 中提取 Answer/Conclusion 后内容 ----------
    def extract_after_Answer(text: str):
        """
        匹配模式：
        Answer: XXX
        **Answer:** XXX
        Conclusion: XXX
        """
        if not text:
            return text
        pattern = r'(\*?\*?(Answer|Conclusion)\*?\*?)[:：]?\s*([\s\S]+)$'
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if not m:
            return text
        # group(3) 是 Answer/Conclusion 后面的部分
        ans_part = m.group(3).strip()
        ans_part = re.sub(r'[\*\n]', '', ans_part).strip()
        return ans_part

    tail_after_ans = extract_after_Answer(tail)
    if debug:
        print(f"[After Answer] {tail_after_ans}")

    # ---------- Step 4: 在 Answer 片段中优先解析字母序号 ----------
    def try_parse_letter_choice(text: str):
        """
        尝试从文本中解析一个字母选项：
        形如: A / [A] / (A) / option A / choice A 等
        只返回单个大写字母 A-J
        """
        if not text:
            return None
        # 去掉多余符号
        cleaned = re.sub(r'[\[\]\(\)\.\,]', ' ', text)
        cleaned = cleaned.upper()

        # 明确的单字母匹配，带边界
        m = re.search(r'\b([A-J])\b', cleaned)
        if m:
            return m.group(1)
        return None

    # 4.1 先看 Answer/Conclusion 后面的内容
    letter = try_parse_letter_choice(tail_after_ans)
    if letter is None:
        # 4.2 如果没找到，再在 tail 全文里找一次
        letter = try_parse_letter_choice(tail)

    if letter is not None:
        if letter in charas[:len(opts)]:
            idx = charas[:len(opts)].index(letter)
            if debug:
                print(f"[Letter Choice] Parsed letter '{letter}' -> idx {idx}")
            return idx

    # ---------- Step 5: 在 tail 中直接匹配选项文本 ----------
    def find_option(options, sentence: str):
        """
        在 sentence 中直接匹配选项字符串（忽略大小写）。
        options: 例如 ['A. dog', 'B. cat'] 或 ['dog','cat']
        返回: (idx, True) 或 (None, False)
        """
        if not sentence:
            return None, False

        sentence_lower = sentence.lower()
        options_lower = [opt.lower() for opt in options]

        # 优先匹配更长的选项，避免子串误匹配
        sorted_indices = sorted(range(len(options_lower)),
                                key=lambda i: len(options_lower[i]),
                                reverse=True)

        for i in sorted_indices:
            opt = options_lower[i]
            if not opt:
                continue
            # 尝试添加边界，防止部分词误匹配
            # 如果选项本身含有标点或空格，就直接用子串匹配
            if re.search(r'\w', opt) and ' ' not in opt:
                # 单词选项，加 word boundary
                pattern = r'\b' + re.escape(opt) + r'\b'
                if re.search(pattern, sentence_lower):
                    return i, True
            else:
                # 短语选项，直接 in
                if opt in sentence_lower:
                    return i, True

        return None, False

    idx, matched = find_option(opts, tail_after_ans)
    if not matched:
        idx, matched = find_option(opts, tail)
    if matched:
        if debug:
            print(f"[Text Match] Matched option: idx={idx}, opt={opts[idx]}")
        return idx

    # ---------- Step 6: 回退相似度匹配（只用 tail） ----------
    # 可选：再对 "is" 后面的片段做一次裁剪，帮助 SBERT 更专注
    def extract_after_is(text: str):
        if not text:
            return text
        m = re.search(r'\bis[:]? \s*(.+)', text, flags=re.IGNORECASE)
        return m.group(1).strip() if m else text

    tail_for_sim = extract_after_is(tail_after_ans)
    if debug:
        print(f"[Tail for sim] {tail_for_sim}")

    # 如果 SBERT 或 SentBERT 未定义，这里提前报错更清晰
    try:
        sims_tail = [SBERT_similarity(str(opt), tail_for_sim, SentBERT) for opt in opts]
    except NameError as e:
        if debug:
            print("SBERT_similarity or SentBERT is not defined:", e)
        # 没有相似度模型时，直接放弃（或返回 -1）
        return -1

    sims_tail = np.array(sims_tail, dtype=float)
    best_idx = int(np.argmax(sims_tail))
    confidence = float(np.max(sims_tail))

    if debug:
        print(f"[Sim Tail] sims={sims_tail}, best_idx={best_idx}, conf={confidence:.3f}")

    if mode != 'sim' and confidence < thre:
        if debug:
            print(f"[Low Confidence] conf={confidence:.3f} < {thre}, return -1; tail={tail_for_sim}")
        return -1

    return best_idx
# ...existing code...

In [69]:
resp = multi_cot_responses[1][0][1]
resp = """khhkhk Answer: [B]. """
opts = ds[option_s][1]
# utils.Answer_Extraction_Pipeline(resp, opts, debug=True, mode='sim', thre=0.9)
Answer_Extraction_Pipeline(resp, opts, debug=True, mode='sim', thre=0.9)

[Tail] khhkhk Answer: [B].
[After Answer] [B].
[Letter Choice] Parsed letter 'B' -> idx 1


1

In [ ]:
import importlib
importlib.reload(utils)

In [ ]:
# old ae
for MODEL_LIST, settings in zip(MODEL_LIST_ALL, settings_all):
    run(MODEL_LIST=MODEL_LIST, settings=settings)
    


16 16
qwen3-235b-a22b-thinking-2507 cot_trigger
Processing qwen3-235b-a22b-thinking-2507 15 ...
Processing qwen3-235b-a22b-thinking-2507 14 ...
Processing qwen3-235b-a22b-thinking-2507 13 ...
Processing qwen3-235b-a22b-thinking-2507 12 ...
Processing qwen3-235b-a22b-thinking-2507 11 ...
Processing qwen3-235b-a22b-thinking-2507 10 ...
Processing qwen3-235b-a22b-thinking-2507 9 ...
Processing qwen3-235b-a22b-thinking-2507 8 ...
Processing qwen3-235b-a22b-thinking-2507 7 ...
Processing qwen3-235b-a22b-thinking-2507 6 ...
Processing qwen3-235b-a22b-thinking-2507 5 ...
Processing qwen3-235b-a22b-thinking-2507 4 ...
Processing qwen3-235b-a22b-thinking-2507 3 ...
Processing qwen3-235b-a22b-thinking-2507 2 ...
Processing qwen3-235b-a22b-thinking-2507 1 ...
Processing qwen3-235b-a22b-thinking-2507 0 ...
qwen3-30b-a3b-thinking-2507 cot_trigger
Processing qwen3-30b-a3b-thinking-2507 15 ...
Processing qwen3-30b-a3b-thinking-2507 14 ...
Processing qwen3-30b-a3b-thinking-2507 13 ...
Processing qwen3

,qwen3-235b-a22b-thinking-2507,qwen3-30b-a3b-thinking-2507,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.97722,0.971749,0.971749,0.97722,0.974484,0.988636,0.989899


16 16
qwen3-235b-a22b-thinking-2507 cot_trigger
Processing qwen3-235b-a22b-thinking-2507 15 ...
Processing qwen3-235b-a22b-thinking-2507 14 ...
Processing qwen3-235b-a22b-thinking-2507 13 ...
Processing qwen3-235b-a22b-thinking-2507 12 ...
Processing qwen3-235b-a22b-thinking-2507 11 ...
Processing qwen3-235b-a22b-thinking-2507 10 ...
Processing qwen3-235b-a22b-thinking-2507 9 ...
Processing qwen3-235b-a22b-thinking-2507 8 ...
Processing qwen3-235b-a22b-thinking-2507 7 ...
Processing qwen3-235b-a22b-thinking-2507 6 ...
Processing qwen3-235b-a22b-thinking-2507 5 ...
Processing qwen3-235b-a22b-thinking-2507 4 ...
Processing qwen3-235b-a22b-thinking-2507 3 ...
Processing qwen3-235b-a22b-thinking-2507 2 ...
Processing qwen3-235b-a22b-thinking-2507 1 ...
Processing qwen3-235b-a22b-thinking-2507 0 ...
qwen3-4b cot_trigger
Processing qwen3-4b 15 ...
Processing qwen3-4b 14 ...
Processing qwen3-4b 13 ...
Processing qwen3-4b 12 ...
Processing qwen3-4b 11 ...
Processing qwen3-4b 10 ...
Processing 

,qwen3-235b-a22b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.97722,0.959991,0.959991,0.97722,0.968605,0.975589,0.986953


16 16
qwen3-30b-a3b-thinking-2507 cot_trigger
Processing qwen3-30b-a3b-thinking-2507 15 ...
Processing qwen3-30b-a3b-thinking-2507 14 ...
Processing qwen3-30b-a3b-thinking-2507 13 ...
Processing qwen3-30b-a3b-thinking-2507 12 ...
Processing qwen3-30b-a3b-thinking-2507 11 ...
Processing qwen3-30b-a3b-thinking-2507 10 ...
Processing qwen3-30b-a3b-thinking-2507 9 ...
Processing qwen3-30b-a3b-thinking-2507 8 ...
Processing qwen3-30b-a3b-thinking-2507 7 ...
Processing qwen3-30b-a3b-thinking-2507 6 ...
Processing qwen3-30b-a3b-thinking-2507 5 ...
Processing qwen3-30b-a3b-thinking-2507 4 ...
Processing qwen3-30b-a3b-thinking-2507 3 ...
Processing qwen3-30b-a3b-thinking-2507 2 ...
Processing qwen3-30b-a3b-thinking-2507 1 ...
Processing qwen3-30b-a3b-thinking-2507 0 ...
qwen3-4b cot_trigger
Processing qwen3-4b 15 ...
Processing qwen3-4b 14 ...
Processing qwen3-4b 13 ...
Processing qwen3-4b 12 ...
Processing qwen3-4b 11 ...
Processing qwen3-4b 10 ...
Processing qwen3-4b 9 ...
Processing qwen3-4b

,qwen3-30b-a3b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.971749,0.959991,0.959991,0.971749,0.96587,0.978956,0.986953


16 16
qwen3-235b-a22b-thinking-2507 cot_trigger
Processing qwen3-235b-a22b-thinking-2507 15 ...
Processing qwen3-235b-a22b-thinking-2507 14 ...
Processing qwen3-235b-a22b-thinking-2507 13 ...
Processing qwen3-235b-a22b-thinking-2507 12 ...
Processing qwen3-235b-a22b-thinking-2507 11 ...
Processing qwen3-235b-a22b-thinking-2507 10 ...
Processing qwen3-235b-a22b-thinking-2507 9 ...
Processing qwen3-235b-a22b-thinking-2507 8 ...
Processing qwen3-235b-a22b-thinking-2507 7 ...
Processing qwen3-235b-a22b-thinking-2507 6 ...
Processing qwen3-235b-a22b-thinking-2507 5 ...
Processing qwen3-235b-a22b-thinking-2507 4 ...
Processing qwen3-235b-a22b-thinking-2507 3 ...
Processing qwen3-235b-a22b-thinking-2507 2 ...
Processing qwen3-235b-a22b-thinking-2507 1 ...
Processing qwen3-235b-a22b-thinking-2507 0 ...
qwen3-30b-a3b-thinking-2507 cot_trigger
Processing qwen3-30b-a3b-thinking-2507 15 ...
Processing qwen3-30b-a3b-thinking-2507 14 ...
Processing qwen3-30b-a3b-thinking-2507 13 ...
Processing qwen3

,qwen3-235b-a22b-thinking-2507,qwen3-30b-a3b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.97722,0.971749,0.959991,0.959991,0.97722,0.969653,0.989057,0.989057


In [5]:
# new ae
for MODEL_LIST, settings in zip(MODEL_LIST_ALL, settings_all):
    run(MODEL_LIST=MODEL_LIST, settings=settings)
    


16 16
qwen3-235b-a22b-thinking-2507 cot_trigger
Processing qwen3-235b-a22b-thinking-2507 15 ...
Processing qwen3-235b-a22b-thinking-2507 14 ...
Processing qwen3-235b-a22b-thinking-2507 13 ...
Processing qwen3-235b-a22b-thinking-2507 12 ...
Processing qwen3-235b-a22b-thinking-2507 11 ...
Processing qwen3-235b-a22b-thinking-2507 10 ...
Processing qwen3-235b-a22b-thinking-2507 9 ...
Processing qwen3-235b-a22b-thinking-2507 8 ...
Processing qwen3-235b-a22b-thinking-2507 7 ...
Processing qwen3-235b-a22b-thinking-2507 6 ...
Processing qwen3-235b-a22b-thinking-2507 5 ...
Processing qwen3-235b-a22b-thinking-2507 4 ...
Processing qwen3-235b-a22b-thinking-2507 3 ...
Processing qwen3-235b-a22b-thinking-2507 2 ...
Processing qwen3-235b-a22b-thinking-2507 1 ...
Processing qwen3-235b-a22b-thinking-2507 0 ...
qwen3-30b-a3b-thinking-2507 cot_trigger
Processing qwen3-30b-a3b-thinking-2507 15 ...
Processing qwen3-30b-a3b-thinking-2507 14 ...
Processing qwen3-30b-a3b-thinking-2507 13 ...
Processing qwen3

,qwen3-235b-a22b-thinking-2507,qwen3-30b-a3b-thinking-2507,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.97722,0.971433,0.971433,0.97722,0.974327,0.989478,0.989478


16 16
qwen3-235b-a22b-thinking-2507 cot_trigger
Processing qwen3-235b-a22b-thinking-2507 15 ...
Processing qwen3-235b-a22b-thinking-2507 14 ...
Processing qwen3-235b-a22b-thinking-2507 13 ...
Processing qwen3-235b-a22b-thinking-2507 12 ...
Processing qwen3-235b-a22b-thinking-2507 11 ...
Processing qwen3-235b-a22b-thinking-2507 10 ...
Processing qwen3-235b-a22b-thinking-2507 9 ...
Processing qwen3-235b-a22b-thinking-2507 8 ...
Processing qwen3-235b-a22b-thinking-2507 7 ...
Processing qwen3-235b-a22b-thinking-2507 6 ...
Processing qwen3-235b-a22b-thinking-2507 5 ...
Processing qwen3-235b-a22b-thinking-2507 4 ...
Processing qwen3-235b-a22b-thinking-2507 3 ...
Processing qwen3-235b-a22b-thinking-2507 2 ...
Processing qwen3-235b-a22b-thinking-2507 1 ...
Processing qwen3-235b-a22b-thinking-2507 0 ...
qwen3-4b cot_trigger
Processing qwen3-4b 15 ...
Processing qwen3-4b 14 ...
Processing qwen3-4b 13 ...
Processing qwen3-4b 12 ...
Processing qwen3-4b 11 ...
Processing qwen3-4b 10 ...
Processing 

,qwen3-235b-a22b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.977141,0.960069,0.960069,0.977141,0.968605,0.977273,0.989057


16 16
qwen3-30b-a3b-thinking-2507 cot_trigger
Processing qwen3-30b-a3b-thinking-2507 15 ...
Processing qwen3-30b-a3b-thinking-2507 14 ...
Processing qwen3-30b-a3b-thinking-2507 13 ...
Processing qwen3-30b-a3b-thinking-2507 12 ...
Processing qwen3-30b-a3b-thinking-2507 11 ...
Processing qwen3-30b-a3b-thinking-2507 10 ...
Processing qwen3-30b-a3b-thinking-2507 9 ...
Processing qwen3-30b-a3b-thinking-2507 8 ...
Processing qwen3-30b-a3b-thinking-2507 7 ...
Processing qwen3-30b-a3b-thinking-2507 6 ...
Processing qwen3-30b-a3b-thinking-2507 5 ...
Processing qwen3-30b-a3b-thinking-2507 4 ...
Processing qwen3-30b-a3b-thinking-2507 3 ...
Processing qwen3-30b-a3b-thinking-2507 2 ...
Processing qwen3-30b-a3b-thinking-2507 1 ...
Processing qwen3-30b-a3b-thinking-2507 0 ...
qwen3-4b cot_trigger
Processing qwen3-4b 15 ...
Processing qwen3-4b 14 ...
Processing qwen3-4b 13 ...
Processing qwen3-4b 12 ...
Processing qwen3-4b 11 ...
Processing qwen3-4b 10 ...
Processing qwen3-4b 9 ...
Processing qwen3-4b

,qwen3-30b-a3b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.971459,0.960122,0.960122,0.971459,0.965791,0.975168,0.986953


16 16
qwen3-235b-a22b-thinking-2507 cot_trigger
Processing qwen3-235b-a22b-thinking-2507 15 ...
Processing qwen3-235b-a22b-thinking-2507 14 ...
Processing qwen3-235b-a22b-thinking-2507 13 ...
Processing qwen3-235b-a22b-thinking-2507 12 ...
Processing qwen3-235b-a22b-thinking-2507 11 ...
Processing qwen3-235b-a22b-thinking-2507 10 ...
Processing qwen3-235b-a22b-thinking-2507 9 ...
Processing qwen3-235b-a22b-thinking-2507 8 ...
Processing qwen3-235b-a22b-thinking-2507 7 ...
Processing qwen3-235b-a22b-thinking-2507 6 ...
Processing qwen3-235b-a22b-thinking-2507 5 ...
Processing qwen3-235b-a22b-thinking-2507 4 ...
Processing qwen3-235b-a22b-thinking-2507 3 ...
Processing qwen3-235b-a22b-thinking-2507 2 ...
Processing qwen3-235b-a22b-thinking-2507 1 ...
Processing qwen3-235b-a22b-thinking-2507 0 ...
qwen3-30b-a3b-thinking-2507 cot_trigger
Processing qwen3-30b-a3b-thinking-2507 15 ...
Processing qwen3-30b-a3b-thinking-2507 14 ...
Processing qwen3-30b-a3b-thinking-2507 13 ...
Processing qwen3

,qwen3-235b-a22b-thinking-2507,qwen3-30b-a3b-thinking-2507,qwen3-4b,min_acc,max_acc,avg_acc,voting_acc,ent_no_prior_acc
ARC-Easy,0.977246,0.97138,0.959912,0.959912,0.977246,0.969513,0.989057,0.989899
